# Weight Stage A — TPU v5e-8 fallback

Chạy 5 cấu hình × seed 42 trên TPU 8 chip. Không dùng GPU quota.
Smoke 2 step trước, sau đó full 5M token/run, 625 step.


In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers>=4.51','peft','kagglehub','huggingface_hub'], check=False)
import torch, torch_xla
import torch_xla.core.xla_model as xm
print('torch', torch.__version__, 'xla', torch_xla.__version__)
print('device', xm.xla_device())


In [ ]:
from huggingface_hub import snapshot_download
root = snapshot_download('ntquang0410/tiki-4model-eval', repo_type='dataset',
                         local_dir='/kaggle/working/tiki-4model-eval')
print('dataset', root)


In [ ]:
import subprocess, sys
from pathlib import Path
root = Path('/kaggle/working/tiki-4model-eval')
common = [sys.executable, str(root/'code/tpu_weight_train.py'),
          '--eval-dir', str(root/'data/weight_eval')]
subprocess.run(common + ['--out','/kaggle/working/smoke_runs','--smoke'], check=True)
print('SMOKE OK')


In [ ]:
import subprocess, sys, time
from pathlib import Path
root = Path('/kaggle/working/tiki-4model-eval')
started=time.time()
subprocess.run([sys.executable, str(root/'code/tpu_weight_train.py'),
                '--eval-dir', str(root/'data/weight_eval'),
                '--out','/kaggle/working/weight_runs','--seed','42'], check=True)
print('DONE', round((time.time()-started)/60,1), 'min')


In [ ]:
from pathlib import Path
for p in sorted(Path('/kaggle/working/weight_runs').glob('hyp_*.parquet')):
    print(p.name, p.stat().st_size)
